In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# تحميل البيانات
df = pd.read_csv("Final_Updates.csv")

# تحويل عمود 'Datetime' إلى نوع بيانات datetime
df['Datetime'] = pd.to_datetime(df['Datetime'])

# استخراج الميزات الزمنية
df['Month'] = df['Datetime'].dt.month
df['Day'] = df['Datetime'].dt.day
df['Hour'] = df['Datetime'].dt.hour
df['Minute'] = df['Datetime'].dt.minute
df['Weekday'] = df['Datetime'].dt.day_name()

# ترميز الأعمدة الفئوية
label_encoder_city = LabelEncoder()
df['City'] = label_encoder_city.fit_transform(df['City'])

label_encoder_congestion = LabelEncoder()
df['CongestionLevel'] = label_encoder_congestion.fit_transform(df['CongestionLevel'])

label_encoder_weekday = LabelEncoder()
df['Weekday'] = label_encoder_weekday.fit_transform(df['Weekday'])

# تحديث قائمة الميزات لتشمل الميزات الجديدة
features = [
    'TrafficIndexLive_norm', 'JamsCount_norm', 'TrafficIndexWeekAgo_norm',
    'TravelTimeHistoric_norm', 'TravelTimeLive_norm',
    'Hour', 'Weekday', 'IsWeekend', 'Year', 'Month', 'Day', 'Minute'
]

# اختيار الميزات والهدف
X = df[features]
y = df['CongestionLevel']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# تدريب نموذج Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, 'rf_congestion_model.pkl')

# حساب التنبؤات
y_pred = rf_model.predict(X_test)

# حساب الدقة
accuracy = accuracy_score(y_test, y_pred)

# تحديد أسماء الفئات النصية
target_names = label_encoder_congestion.classes_

# طباعة تقرير التصنيف
classification_report_result = classification_report(y_test, y_pred, target_names=target_names.astype(str))  # تحويل القيم إلى نصوص

print(f"Model Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report_result)


Model Accuracy: 0.9713250517598344
Classification Report:
              precision    recall  f1-score   support

        High       0.99      0.98      0.99      8071
         Low       0.94      0.97      0.95       477
    Moderate       0.86      0.89      0.88      1112

    accuracy                           0.97      9660
   macro avg       0.93      0.95      0.94      9660
weighted avg       0.97      0.97      0.97      9660



In [4]:
xgb_model = joblib.load('rf_congestion_model.pkl')


trained_features = [
    'TrafficIndexLive_norm', 
    'JamsCount_norm', 
    'TrafficIndexWeekAgo_norm', 
    'TravelTimeHistoric_norm', 
    'TravelTimeLive_norm',
    'Hour', 
    'Weekday', 
    'IsWeekend', 
    'Year', 
    'Month', 
    'Day', 
    'Minute'
]

In [5]:
def normalize_input(user_input):
    user_input['TrafficIndexLive_norm'] = user_input['TrafficIndexLive'] / 99
    user_input['JamsCount_norm'] = user_input['JamsCount'] / 883
    user_input['TrafficIndexWeekAgo_norm'] = user_input['TrafficIndexWeekAgo'] / 99
    user_input['TravelTimeHistoric_norm'] = user_input['TravelTimeHistoric'] / 94.7783212767502
    user_input['TravelTimeLive_norm'] = user_input['TravelTimeLive'] / 130.973103722656
    return user_input


In [6]:
# إدخال المستخدم
def get_user_input():
    return {
        'TrafficIndexLive': float(input("TrafficIndexLive: ")),
        'JamsCount': int(input("JamsCount: ")),
        'TrafficIndexWeekAgo': float(input("TrafficIndexWeekAgo: ")),
        'TravelTimeHistoric': float(input("TravelTimeHistoric: ")),
        'TravelTimeLive': float(input("TravelTimeLive: ")),
        'Hour': int(input("Hour (0-23): ")),
        'Weekday': int(input("Weekday (0=Monday to 6=Sunday): ")),
        'IsWeekend': int(input("IsWeekend (0 or 1): ")),
        'Year': int(input("Year: ")),
        'Month': int(input("Month: ")),
        'Day': int(input("Day: ")),
        'Minute': int(input("Minute: "))
    }



    return {
        'TrafficIndexLive': traffic_index_live,
        'JamsCount': jams_count,
        'TrafficIndexWeekAgo': traffic_index_week_ago,
        'TravelTimeHistoric': travel_time_historic,
        'TravelTimeLive': travel_time_live,
        'Hour': hour,
        'Weekday': weekday,
        'IsWeekend': is_weekend,
        'Year': year,
        'Month': month,
        'Day': day,
        'Minute': minute
    }


In [ ]:
# دالة التنبؤ
def predict_congestion(user_input):
    user_input = normalize_input(user_input)

    # اختيار فقط الميزات المستخدمة في التدريب
    input_filtered = {key: user_input[key] for key in trained_features}
    user_input_df = pd.DataFrame([input_filtered], columns=trained_features)

    # التنبؤ
    prediction = xgb_model.predict(user_input_df)
    predicted_label = label_encoder_congestion.inverse_transform(prediction)
    return predicted_label[0]

# الاستخدام
user_input = get_user_input()
result = predict_congestion(user_input)
print(f"Predicted Congestion Level: {result}")